# Save Targets and App

Loads the **`app`** and **`target`** tables from the `power-<bureau>-prod` PROCESSED S3 layout, one date partition per split, and saves them locally:
```
payment_processing_research_data/<bureau>/<split>/app/part-NNNNN.parquet
payment_processing_research_data/<bureau>/<split>/target/part-NNNNN.parquet
```
Paths come straight from `configs.py`. Target uses the existing `target_path()`; app reuses the same base with `TABLE=target` -> `TABLE=app`.

Needs `s3fs` installed and AWS creds (default profile / `AWS_*` env vars).

In [1]:
1+1

2

In [2]:
import os, sys
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as pds
import s3fs

sys.path.insert(0, os.getcwd())   # configs.py is next to this notebook
import configs
from configs import BUREAUS, DATA_DIR

BUREAUS_TO_RUN = ['equifax', 'experian', 'transunion']
SPLITS         = ['train', 'valid', 'test']
TABLES         = ['app', 'target']
CHUNK_SIZE     = 100_000

_FS = s3fs.S3FileSystem()


def s3_path(cfg, table, split):
    # Built from the bureau's config fields (NOT target_base, which can be stale).
    date = cfg[split + '_date']
    base = ('power-' + cfg['bureau'] + '-prod/NATIONAL/PROCESSED/DATA/'
            'BUREAU='       + cfg['bureau'] +
            '/FORMAT='      + cfg['format'] +
            '/TABLE='       + table +
            '/PULL_NAME='   + cfg['pull_name'] +
            '/FEW_VERSION=' + cfg['few_version'] +
            '/ME_VERSION='  + cfg['me_version'])
    return 's3://' + base + '/DATE_OF_REQUEST=' + date


def _widen(a, b):
    # one type that holds both: null defers to the real type; int+float -> float64;
    # anything else incompatible -> string (read as text, never errors).
    if pa.types.is_null(a):                               return b
    if pa.types.is_null(b):                               return a
    if a == b:                                            return a
    numeric = lambda t: pa.types.is_integer(t) or pa.types.is_floating(t)
    if pa.types.is_integer(a) and pa.types.is_integer(b): return pa.int64()
    if numeric(a) and numeric(b):                         return pa.float64()
    return pa.string()


def read_partition(path):
    # One pd.read_parquet over the whole folder. The part files disagree on some
    # column types across shards (null vs double, int64 vs double, ...), which a
    # plain read / unify_schemas can't reconcile. So we build ONE widened schema
    # covering every file and read with it -- the reader casts each shard up to it.
    ds = pds.dataset(path[len('s3://'):], filesystem=_FS, partitioning='hive')
    merged = {}
    for frag in ds.get_fragments():
        for f in frag.physical_schema:
            merged[f.name] = f.type if f.name not in merged else _widen(merged[f.name], f.type)
    for f in ds.schema:                       # keep hive partition columns
        merged.setdefault(f.name, f.type)
    schema = pa.schema([pa.field(n, t) for n, t in merged.items()])
    return pd.read_parquet(path, schema=schema)


# eyeball every path before pulling anything
for b in BUREAUS_TO_RUN:
    for t in TABLES:
        for s in SPLITS:
            print(s3_path(BUREAUS[b], t, s))

s3://power-equifax-prod/NATIONAL/PROCESSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=app/PULL_NAME=national_1.1/FEW_VERSION=0.1.0/ME_VERSION=v1.9.0.rc1/DATE_OF_REQUEST=2019-03-31
s3://power-equifax-prod/NATIONAL/PROCESSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=app/PULL_NAME=national_1.1/FEW_VERSION=0.1.0/ME_VERSION=v1.9.0.rc1/DATE_OF_REQUEST=2019-06-30
s3://power-equifax-prod/NATIONAL/PROCESSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=app/PULL_NAME=national_1.1/FEW_VERSION=0.1.0/ME_VERSION=v1.9.0.rc1/DATE_OF_REQUEST=2019-12-31
s3://power-equifax-prod/NATIONAL/PROCESSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=target/PULL_NAME=national_1.1/FEW_VERSION=0.1.0/ME_VERSION=v1.9.0.rc1/DATE_OF_REQUEST=2019-03-31
s3://power-equifax-prod/NATIONAL/PROCESSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=target/PULL_NAME=national_1.1/FEW_VERSION=0.1.0/ME_VERSION=v1.9.0.rc1/DATE_OF_REQUEST=2019-06-30
s3://power-equifax-prod/NATIONAL/PROCESSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=target/PULL_NAME=national_1.1/FE

In [3]:
BUREAUS_TO_RUN

['equifax', 'experian', 'transunion']

In [4]:
for bureau in BUREAUS_TO_RUN:
    cfg = BUREAUS[bureau]
    print(cfg)

{'bureau': 'equifax', 'schema': 'EQUIFAX', 'format': 'cms_6', 'pull_name': 'national_1.1', 'few_version': '0.1.0', 'me_version': 'v1.9.0.rc1', 'train_date': '2019-03-31', 'valid_date': '2019-06-30', 'test_date': '2019-12-31', 'trade_base': 'power-equifax-prod/NATIONAL/PARSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=trade/PULL_NAME=national_1', 'trade_fe_base': 'power-equifax-prod/NATIONAL/PROCESSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=trade_fe/PULL_NAME=national_1.1/FEW_VERSION=0.1.0/ME_VERSION=v1.9.9rc1', 'target_base': 'power-equifax-prod/NATIONAL/PROCESSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=target/PULL_NAME=national_1.1/FEW_VERSION=0.1.0/ME_VERSION=v1.9.9rc1'}
{'bureau': 'experian', 'schema': 'EXPERIAN', 'format': 'arf7', 'pull_name': 'national_1.3', 'few_version': '0.0.9', 'me_version': 'v1.8.0rc0', 'train_date': '2019-03-31', 'valid_date': '2019-06-30', 'test_date': '2019-12-31', 'trade_base': 'power-experian-prod/NATIONAL/PARSED/DATA/BUREAU=experian/FORMAT=arf7/TABLE=trade/PU

In [5]:
path = 's3://power-equifax-prod/NATIONAL/PROCESSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=target/PULL_NAME=national_1.1/FEW_VERSION=0.1.0/ME_VERSION=v1.9.0.rc1'

In [ ]:
pd.read_parquet(path)

In [ ]:
1+1

In [ ]:
summary = []
for bureau in BUREAUS_TO_RUN:
    cfg = BUREAUS[bureau]
    for split in SPLITS:
        for table in TABLES:
            path = s3_path(cfg, table, split)
            #print(path)
            out_dir = os.path.join(DATA_DIR, bureau, split, table)
            #print(out_dir)
            os.makedirs(out_dir, exist_ok=True)
            for old in os.listdir(out_dir):
                
                if old.startswith('part-') and old.endswith('.parquet'):
                    print('old')
                    if old.startswith('part-') and old.endswith('.parquet'):
                        os.remove(os.path.join(out_dir, old))
                else:
                    print('no old')

In [3]:
summary = []
for bureau in BUREAUS_TO_RUN:
    cfg = BUREAUS[bureau]
    print('bureau')
    for split in SPLITS:
        for table in TABLES:
            path = s3_path(cfg, table, split)
            print('[' + bureau + '/' + split + '/' + table + '] reading ' + path)
            df = read_partition(path)

            out_dir = os.path.join(DATA_DIR, bureau, split, table)
            print(out_dir)
            os.makedirs(out_dir, exist_ok=True)
            for old in os.listdir(out_dir):
                print(f'removing files')
                if old.startswith('part-') and old.endswith('.parquet'):
                    os.remove(os.path.join(out_dir, old))
            for i in range(0, len(df), CHUNK_SIZE):
                df.iloc[i:i + CHUNK_SIZE].to_parquet(
                    os.path.join(out_dir, 'part-%05d.parquet' % (i // CHUNK_SIZE)), index=False)

            print('   -> %s rows x %d cols -> %s' % (format(len(df), ','), df.shape[1], out_dir))
            summary.append([bureau, split, table, len(df), df.shape[1]])

pd.DataFrame(summary, columns=['bureau', 'split', 'table', 'rows', 'cols'])

bureau
[equifax/train/app] reading s3://power-equifax-prod/NATIONAL/PROCESSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=app/PULL_NAME=national_1.1/FEW_VERSION=0.1.0/ME_VERSION=v1.9.0.rc1/DATE_OF_REQUEST=2019-03-31
/home/jag/payment-processor-research/payment_processing_research_data/equifax/train/app
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
removing files
   -> 2,772,507 rows x 53 cols -> /home/jag/payment-processor-research/payment_processing_research_data/equifax/train/app
[equifax/train/target] reading s3://power-equifax-prod/NATIONAL/PROCESSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=target/PULL_NAME=national_1.1/FEW_VERSION=0.1.

,bureau,split,table,rows,cols
0,equifax,train,app,2772507,53
1,equifax,train,target,2772507,149
2,equifax,valid,app,2843496,53
3,equifax,valid,target,2843496,149
4,equifax,test,app,2646909,53
5,equifax,test,target,2646909,149
6,experian,train,app,2365955,28
7,experian,train,target,2365955,117
8,experian,valid,app,2437409,28
9,experian,valid,target,2437409,117
